# Genre distribution as percentage of songs accross the whole dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import PurePosixPath

genre_wide_df = pd.read_csv("data_reports/genre_dist.csv")

genre_cols = [
    c for c in genre_wide_df.columns
    if c not in ["id", "midi_path"]
]
# Convert genre columns to boolean
genre_bool_df = genre_wide_df.copy()
genre_bool_df[genre_cols] = (
    genre_bool_df[genre_cols]
    .fillna(False)
    .astype(bool)
)

# Collapse to song / track level.
# If any MIDI version of the same track has a genre=True, the track gets that genre=True.
song_level_genres = (
    genre_bool_df
    .groupby("id")[genre_cols]
    .any()
)

# Calculate percentage of songs/tracks with each genre
genre_percentages_song_level = (
    song_level_genres.sum()
    .div(len(song_level_genres))
    .mul(100)
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 8))
ax = sns.barplot(
    x=genre_percentages_song_level.values,
    y=genre_percentages_song_level.index,
)

ax.set_xlabel("Percentage of songs (%)")
ax.set_ylabel("Genre")
ax.set_title("Genre distribution as percentage of songs")
ax.set_xlim(0, max(genre_percentages_song_level.max() * 1.15, 5))

for i, value in enumerate(genre_percentages_song_level.values):
    ax.text(
        value + 0.5,
        i,
        f"{value:.1f}%",
        va="center",
    )

plt.tight_layout()
plt.show()

# Genre distribution accross the splits

In [ ]:
num_midi_files = len(genre_wide_df)
num_songs = genre_wide_df["track_id"].nunique()

print(f"MIDI-file-level rows: {num_midi_files}")
print(f"Unique songs/tracks: {num_songs}")
print(f"Average MIDI files per song: {num_midi_files / num_songs:.2f}")

In [ ]:
import seaborn as sns
import json

json_path = "data_reports/splits/split_report.json"
with open(json_path, "r") as f:
    data = json.load(f)
splits = data["splits"]

rows = []

for split_name, split_data in splits.items():
    genre_dict = split_data["genre_id_proportions"]
    
    for genre, value in genre_dict.items():
        clean_genre = genre.replace("genre:", "")  # remove prefix
        rows.append({
            "genre": clean_genre,
            "split": split_name,
            "proportion": value
        })

df = pd.DataFrame(rows)
heatmap_data = df.pivot(index="genre", columns="split", values="proportion")
heatmap_data = heatmap_data[["train", "val", "test"]]
heatmap_data = heatmap_data.loc[
    heatmap_data.mean(axis=1).sort_values(ascending=True).index
]
plt.figure(figsize=(8, 10))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap="viridis"
)

plt.title("Genre Distribution Across Splits")
plt.xlabel("Split")
plt.ylabel("Genre")

plt.tight_layout()
plt.show()

# Instrument distribution accross the splits

In [ ]:
import pretty_midi as pm
GM_FAMILIES = [
    "Piano", "Chromatic Percussion", "Organ", "Guitar",
    "Bass", "Strings", "Ensemble", "Brass",
    "Reed", "Pipe", "Synth Lead", "Synth Pad",
    "Synth Effects", "Ethnic", "Percussive", "Sound Effects"
]

def program_to_family(prog_num):
    return GM_FAMILIES[prog_num // 8]

def clean_instrument_name(label):
    if label == "instrument:drums":
        return "Drums", "Drums"
    
    inst = pm.program_to_instrument_name(int(label.split("_")[1]))
    return inst, program_to_family(int(label.split("_")[1]))
rows = []

for split_name, split_data in data["splits"].items():
    inst_dict = split_data["instrument_label_counts"]
    total = sum(inst_dict.values())
    
    for inst, count in inst_dict.items():
        name, family = clean_instrument_name(inst)
        
        rows.append({
            "split": split_name,
            "instrument": name,
            "family": family,
            "count": count,
            "proportion": count / total
        })

df = pd.DataFrame(rows)
top_instruments = (
    df.groupby("instrument")["count"]
    .sum()
    .sort_values(ascending=False)
    .head(30)
    .index
)

df_top = df[df["instrument"].isin(top_instruments)]

plt.figure(figsize=(14, 6))

sns.barplot(
    data=df_top,
    x="instrument",
    y="proportion",
    hue="split"
)

plt.xticks(rotation=60, ha="right")
plt.title("Top 30 Instruments Distribution Across Splits")
plt.tight_layout()
plt.show()

# Instrument family distribution accross the splits

In [ ]:
df_family = (
    df.groupby(["split", "family"])["proportion"]
    .sum()
    .reset_index()
)

heatmap_data = df_family.pivot(
    index="family",
    columns="split",
    values="proportion"
).fillna(0)
heatmap_data = heatmap_data[["train", "val", "test"]]
heatmap_data = heatmap_data.loc[
    heatmap_data.mean(axis=1).sort_values(ascending=True).index
]
plt.figure(figsize=(8, 6))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap="viridis"
)

plt.title("Instrument Family Distribution Across Splits")
plt.tight_layout()
plt.show()

# Instrument statistics in the whole dataset

In [ ]:
from collections import Counter
from pathlib import Path
import pretty_midi
from tqdm import tqdm

instrument_counter = Counter()
missing_midi_count = 0
unreadable_midi_count = 0

for midi_path in tqdm(genre_wide_df["midi_path"]):
    midi_path = Path(midi_path)

    # Count missing files without printing each one
    if not midi_path.exists():
        missing_midi_count += 1
        continue

    try:
        midi_data = pretty_midi.PrettyMIDI(str(midi_path))

        instruments_in_file = set()

        for instrument in midi_data.instruments:
            if instrument.is_drum:
                name = "Drums"
            else:
                name = pretty_midi.program_to_instrument_name(instrument.program)

            instruments_in_file.add(name)

        # Counts each instrument at most once per MIDI file
        instrument_counter.update(instruments_in_file)

    except Exception:
        # File exists, but pretty_midi could not parse it
        unreadable_midi_count += 1

print(f"Missing MIDI files: {missing_midi_count}")
print(f"Unreadable MIDI files: {unreadable_midi_count}")
print(f"Successfully processed MIDI files: {len(genre_wide_df) - missing_midi_count - unreadable_midi_count}")

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Top 5 instruments
top_5 = instrument_counter.most_common(5)

top_5_df = pd.DataFrame(top_5, columns=["instrument", "count"])

# Sum of all remaining instruments
others_count = sum(instrument_counter.values()) - top_5_df["count"].sum()

# Add "Others" if there are any remaining instruments
if others_count > 0:
    plot_df = pd.concat([
        top_5_df,
        pd.DataFrame([{"instrument": "Others", "count": others_count}])
    ], ignore_index=True)
else:
    plot_df = top_5_df

# Colors
colors = sns.color_palette("pastel", n_colors=len(plot_df))

# Make "Others" stand out
explode = [0] * len(plot_df)
if others_count > 0:
    explode[-1] = 0.08   # slightly offset the "Others" slice

plt.figure(figsize=(7, 7))

plt.pie(
    plot_df["count"],
    labels=plot_df["instrument"],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    explode=explode,
    wedgeprops={"edgecolor": "white", "linewidth": 1},
)

plt.title("Top 5 Most Frequent Instruments in MIDI Dataset")
plt.axis("equal")
plt.show()